# Omar ELSEMRY HW1 — PAC vs Agnostic PAC

We learn a threshold classifier: $h_t(x) = 1$ if $x > t$, else $0$, with $x \in [0,1]$.

**My choices:** $\theta = 0.3$, $\varepsilon = 0.05$, $\eta = 0.2$ (noise), 500 repetitions, test set of 10 000 points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

THETA = 0.3   # true threshold
EPS = 0.05    # accuracy
ETA = 0.2     # noise rate
SIZES = [5, 10, 20, 50, 100, 200]
REPS = 500

## 1. Data Generation

In [ ]:
def generate_data(m, eta=0.0):
    x = np.random.uniform(0, 1, m)      # uniform points in [0, 1]
    y = (x > THETA).astype(int)         # true labels
    flip = np.random.rand(m) < eta      # flip each label with probability eta
    y[flip] = 1 - y[flip]
    return x, y

datasets = {m: generate_data(m) for m in SIZES}

# Plot one dataset
x, y = datasets[50]
plt.scatter(x, y, c=y, cmap="coolwarm")
plt.axvline(THETA, color="green", linestyle="--", label="true theta")
plt.xlabel("x"); plt.ylabel("y"); plt.title("Dataset with m = 50"); plt.legend()
plt.show()

## 2. ERM
**Strategy:** the training error only changes when $t$ passes a training point,
so we try $t = 0$ and every training point $x_i$, and keep the $t$ with the smallest training error.

In [ ]:
def error(t, x, y):
    return np.mean((x > t).astype(int) != y)

def erm(x, y):
    candidates = np.append(0, x)
    errors = [error(t, x, y) for t in candidates]
    return candidates[np.argmin(errors)]

## 3. Performance Evaluation

In [ ]:
x_test, y_test = generate_data(10_000)   # large test set to estimate the true error

print("m    learned t   train error   true error")
for m in SIZES:
    x, y = datasets[m]
    t = erm(x, y)
    print(m, "  ", round(t, 3), "      ", error(t, x, y), "       ", error(t, x_test, y_test))

**Comment:** the training error is always 0. The true error is a bit higher, and it gets smaller when $m$ grows.

## 4. PAC Learning (no noise)

In [ ]:
def experiment(eta):
    x_test, y_test = generate_data(10_000, eta)
    prob_bad, train_err, true_err = [], [], []
    for m in SIZES:
        tr, te = [], []
        for _ in range(REPS):
            x, y = generate_data(m, eta)
            t = erm(x, y)
            tr.append(error(t, x, y))
            te.append(error(t, x_test, y_test))
        tr, te = np.array(tr), np.array(te)
        train_err.append(tr.mean())
        true_err.append(te.mean())
        prob_bad.append(np.mean(te - eta > EPS))   # P(L(h) > eps); for noise, compared to the best error eta
    return prob_bad, train_err, true_err

pac = experiment(eta=0.0)

plt.plot(SIZES, pac[0], "o-")
plt.xlabel("m"); plt.ylabel("P(L(h) > eps)"); plt.title("PAC: probability of a bad hypothesis")
plt.show()

**Comment:** the probability that the error is larger than $\varepsilon$ goes to 0 when $m$ grows.
This is what PAC learning says: with enough data, ERM is almost always almost correct.

## 5. Agnostic PAC (with noise $\eta = 0.2$)
With noise, even the true threshold makes 20% errors. So we check if the error is close to this best error: $\Pr(L(h) - \eta > \varepsilon)$.

In [ ]:
agn = experiment(eta=ETA)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(SIZES, pac[0], "o-", label="no noise")
ax[0].plot(SIZES, agn[0], "s-", label="with noise")
ax[0].set_xlabel("m"); ax[0].set_title("Probability of a bad hypothesis"); ax[0].legend()

ax[1].plot(SIZES, agn[1], "o-", label="train error")
ax[1].plot(SIZES, agn[2], "s-", label="true error")
ax[1].axhline(ETA, color="gray", linestyle="--", label="eta = 0.2")
ax[1].set_xlabel("m"); ax[1].set_title("Errors with noise"); ax[1].legend()

plt.show()

**Can the training error still reach zero?** No (except sometimes for very small $m$). The noisy labels cannot all be fitted, so the training error stays around 0.2.

**Does more data still help?** Yes, but the true error goes down only to 0.2, not to 0. The probability of a bad hypothesis also goes to 0, but more slowly than without noise.

**Link with the course:** in PAC, a perfect hypothesis exists, so we can get error close to 0.
In Agnostic PAC, no perfect hypothesis exists, so we can only get close to the **best error in the class** ($\eta$), and we need more examples.

## Conclusion
- **PAC (no noise):** training error = 0, true error → 0 when $m$ grows.
- **Agnostic PAC (noise):** training error ≈ $\eta$, true error → $\eta$ (never 0), and learning is slower.